# Decision Tree

**Idea:** the tree asks yes/no questions like *"is feature 12 <= 0.35?"*.
The answer sends the example left or right, to the next question. At the end (a **leaf**) the tree gives a class.

We build the tree from the training data. At each step we choose the question that best separates the classes.

What we test:
* **max_depth** – how many questions in a row (a deep tree can memorise the training data = overfitting)
* **min_samples_leaf** – the smallest number of rows allowed in a leaf
* **criterion** – Gini or entropy

All tree code is written by us (NumPy only, no sklearn).

## 1. Setup

In [ ]:
import os
import sys

# The notebook is in a sub-folder. Go to the project folder (where common.py is).
if not os.path.exists("common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common   # our shared file: load data, split, metrics, Kaggle file

common.set_seed(42)
os.makedirs("decision_tree/outputs", exist_ok=True)

## The tree code

**Impurity** – how mixed the classes are in a group of rows (0 = only one class):
* Gini: `1 - sum(p_c^2)`
* Entropy: `-sum(p_c * log2(p_c))`

`p_c` = share of class `c` in the group.

**Building the tree** (function `build`):
1. If we must stop (max depth reached, too few rows, or only one class left) → make a **leaf**. The leaf predicts the most common class.
2. Otherwise try many questions *"feature f <= threshold t ?"*. For each one, split the rows into left (yes) and right (no).
3. **Gain** = impurity before − weighted impurity after. Keep the question with the biggest gain.
4. Build the left child and the right child the same way (recursion).

**Faster search:** we do not try every value as a threshold.
* For every feature we only try 31 thresholds (its quantiles).
* At the start, every value is replaced by its **bin number** (which interval between thresholds it falls in).
* At each node, one `np.bincount` makes a table: for every feature and every bin, how many rows of each class.
* A cumulative sum over the bins gives the class counts on the left side for every threshold.
* So we get the gain of every (feature, threshold) question at once, and keep the biggest.

**Prediction:** start at the root, answer the question, go left or right, until a leaf.

In [ ]:
def gini(counts):
    """counts[..., c] = number of rows of class c in a group (last axis = classes).
    Returns one impurity per group."""
    totals = counts.sum(axis=-1, keepdims=True)
    p = counts / np.maximum(totals, 1)
    return 1 - (p ** 2).sum(axis=-1)


def entropy(counts):
    totals = counts.sum(axis=-1, keepdims=True)
    p = counts / np.maximum(totals, 1)
    log_p = np.log2(np.where(p > 0, p, 1))      # log2(1) = 0, so empty classes add nothing
    return -(p * log_p).sum(axis=-1)


IMPURITY = {"gini": gini, "entropy": entropy}


def make_thresholds(X, n_thresholds):
    """For every feature: n_thresholds candidate thresholds (quantiles). Shape (n_features, n_thresholds)."""
    q = np.linspace(0, 1, n_thresholds + 2)[1:-1]
    return np.quantile(X, q, axis=0).T


def to_bins(X, thresholds):
    """Bin number of every value = how many thresholds are smaller than it.
    So: value <= thresholds[f, t]  is the same as  bin <= t."""
    bins = np.zeros(X.shape, dtype=int)
    for f in range(X.shape[1]):
        bins[:, f] = np.searchsorted(thresholds[f], X[:, f])
    return bins


class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 criterion="gini", max_features=None, n_thresholds=31, seed=42):
        self.max_depth = max_depth                  # None = no limit
        self.min_samples_split = min_samples_split  # a node needs at least this many rows to be split
        self.min_samples_leaf = min_samples_leaf    # each child must keep at least this many rows
        self.criterion = criterion                  # "gini" or "entropy"
        self.max_features = max_features            # None = try all features (Random Forest: a random few)
        self.n_thresholds = n_thresholds
        self.rng = np.random.default_rng(seed)

    # ---------------- training ----------------
    def fit(self, X, y, n_classes):
        self.n_classes = n_classes
        self.impurity = IMPURITY[self.criterion]
        self.thresholds = make_thresholds(X, self.n_thresholds)
        bins = to_bins(X, self.thresholds)
        self.root = self.build(bins, y, depth=0)
        return self

    def build(self, bins, y, depth):
        counts = np.bincount(y, minlength=self.n_classes)
        leaf = {"leaf": True, "label": int(np.argmax(counts))}     # most common class

        if len(y) < self.min_samples_split:
            return leaf
        if self.max_depth is not None and depth >= self.max_depth:
            return leaf
        if counts.max() == len(y):                                  # only one class left
            return leaf

        feature, t = self.best_split(bins, y, counts)
        if feature is None:                                         # no split makes it better
            return leaf

        go_left = bins[:, feature] <= t
        return {"leaf": False,
                "feature": feature,
                "threshold": self.thresholds[feature, t],
                "left": self.build(bins[go_left], y[go_left], depth + 1),
                "right": self.build(bins[~go_left], y[~go_left], depth + 1)}

    def best_split(self, bins, y, counts):
        n_rows, n_features = bins.shape
        n_bins = self.n_thresholds + 1
        C = self.n_classes
        parent = self.impurity(counts)                              # impurity before the split

        features = np.arange(n_features)
        if self.max_features is not None:                           # Random Forest: a few random features
            features = self.rng.choice(n_features, self.max_features, replace=False)
        F = len(features)

        # table[f, b, c] = number of rows of class c that fall in bin b of feature f.
        # To count everything with ONE np.bincount, every (feature, bin, class) gets its own number:
        #     cell = (f * n_bins + b) * C + c
        cell = (np.arange(F) * n_bins + bins[:, features]) * C + y[:, None]
        table = np.bincount(cell.ravel(), minlength=F * n_bins * C).reshape(F, n_bins, C)

        # Threshold t sends bins 0..t to the left -> cumulative sum over the bins
        left = np.cumsum(table, axis=1)[:, :-1, :]      # shape (F, n_thresholds, C)
        right = counts - left
        n_left = left.sum(axis=2)
        n_right = right.sum(axis=2)

        after = (n_left * self.impurity(left) + n_right * self.impurity(right)) / n_rows
        gain = parent - after                           # shape (F, n_thresholds): one gain per question
        too_small = (n_left < self.min_samples_leaf) | (n_right < self.min_samples_leaf)
        gain[too_small] = 0

        f, t = np.unravel_index(np.argmax(gain), gain.shape)
        if gain[f, t] <= 1e-12:                         # no question makes the groups purer
            return None, None
        return features[f], t

    # ---------------- prediction ----------------
    def predict_one(self, x):
        node = self.root
        while not node["leaf"]:
            if x[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["label"]

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])


def count_nodes(node):
    if node["leaf"]:
        return 1
    return 1 + count_nodes(node["left"]) + count_nodes(node["right"])


def tree_depth(node):
    if node["leaf"]:
        return 0
    return 1 + max(tree_depth(node["left"]), tree_depth(node["right"]))

In [ ]:
def print_tree(node, depth=0, names=None):
    """Print a small tree, one line per node."""
    pad = "    " * depth
    if node["leaf"]:
        print(f"{pad}-> class {node['label']}")
        return
    name = names[node["feature"]] if names else f"feature {node['feature']}"
    print(f"{pad}{name} <= {node['threshold']:.2f} ?")
    print_tree(node["left"], depth + 1, names)
    print(f"{pad}else:")
    print_tree(node["right"], depth + 1, names)

## 2. Small example

200 points with 2 features. The true rule: class 1 if `x0 + x1 > 1`.
A tree can only cut along one feature at a time, so it makes a "staircase" around the diagonal line.

In [ ]:
rng = np.random.default_rng(0)
X_toy = rng.random((200, 2))
y_toy = (X_toy[:, 0] + X_toy[:, 1] > 1).astype(int)

for depth in [1, 2, 4]:
    toy = DecisionTree(max_depth=depth).fit(X_toy, y_toy, n_classes=2)
    print(f"--- max_depth = {depth}   training accuracy {common.accuracy(y_toy, toy.predict(X_toy)):.3f}")
    if depth <= 2:
        print_tree(toy.root, names=["x0", "x1"])

# What the depth-4 tree predicts everywhere
xx, yy = np.meshgrid(np.linspace(0, 1, 100), np.linspace(0, 1, 100))
grid = np.c_[xx.ravel(), yy.ravel()]
plt.figure(figsize=(4.5, 4))
plt.contourf(xx, yy, toy.predict(grid).reshape(xx.shape), alpha=0.3, cmap="coolwarm")
plt.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap="coolwarm", s=12)
plt.plot([0, 1], [1, 0], "k--", label="true line")
plt.title("Depth-4 tree")
plt.legend()
plt.show()

## 3. Load the data

In [ ]:
X_image, y_image, X_image_test, image_test_ids = common.load_task("image")
X_sent, y_sent, X_sent_test, sent_test_ids = common.load_task("sentiment")

DATA = {"image": (X_image, y_image), "sentiment": (X_sent, y_sent)}
TEST = {"image": (X_image_test, image_test_ids), "sentiment": (X_sent_test, sent_test_ids)}
N_CLASSES = common.N_CLASSES    # {"image": 50, "sentiment": 3}


def prepare(task):
    """Same 80/20 split as every model. No preprocessing: a tree compares one feature
    with a threshold, so scaling a feature only moves the threshold."""
    X, y = DATA[task]
    train_idx, val_idx = common.stratified_split(y)
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]

## 4. Experiments

### 4.1 Maximum depth

* Small depth: train and validation both low → **underfitting**.
* Big depth: train goes to 1.0, validation goes down → **overfitting**.

In [ ]:
results = []
best_trees = {}      # task -> (validation accuracy, settings, tree)


def try_tree(task, **settings):
    X_tr, y_tr, X_va, y_va = prepare(task)
    start = time.time()
    tree = DecisionTree(**settings).fit(X_tr, y_tr, N_CLASSES[task])
    row = {"task": task, "settings": str(settings),
           "train_acc": round(common.accuracy(y_tr, tree.predict(X_tr)), 4),
           "val_acc": round(common.accuracy(y_va, tree.predict(X_va)), 4),
           "nodes": count_nodes(tree.root), "depth": tree_depth(tree.root),
           "seconds": round(time.time() - start, 1)}
    results.append(row)
    print(f"{task:<9} {str(settings):<45} train {row['train_acc']:.3f}   validation {row['val_acc']:.3f}"
          f"   nodes {row['nodes']:6d}   ({row['seconds']}s)")
    if task not in best_trees or row["val_acc"] > best_trees[task][0]:
        best_trees[task] = (row["val_acc"], settings, tree)
    return row


DEPTHS = [2, 5, 8, 12, 16, 20, None]
for task in ["sentiment", "image"]:
    rows = [try_tree(task, max_depth=d) for d in DEPTHS]
    labels = [str(d) for d in DEPTHS]
    plt.figure(figsize=(6, 3.5))
    plt.plot(labels, [r["train_acc"] for r in rows], marker="o", label="train")
    plt.plot(labels, [r["val_acc"] for r in rows], marker="o", label="validation")
    plt.xlabel("max_depth (None = no limit)")
    plt.ylabel("accuracy")
    plt.title(f"{task}: tree depth")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

### 4.2 Minimum rows in a leaf, and Gini vs entropy

We keep the best depth from 4.1. **Change these two numbers if your table says something else.**

In [ ]:
BEST_DEPTH = {"sentiment": 5, "image": None}

for task in ["sentiment", "image"]:
    for leaf in [5, 20, 50]:
        try_tree(task, max_depth=BEST_DEPTH[task], min_samples_leaf=leaf)
    try_tree(task, max_depth=BEST_DEPTH[task], criterion="entropy")

### 4.3 All results

In [ ]:
table = pd.DataFrame(results)
for task in ["sentiment", "image"]:
    t = table[table["task"] == task].sort_values("val_acc", ascending=False)
    t.to_csv(f"decision_tree/outputs/{task}_experiments.csv", index=False)
    display(t)

## 5. Best tree: error analysis

In [ ]:
def show_errors(task, y_true, y_pred):
    cm = common.confusion_matrix(y_true, y_pred, N_CLASSES[task])
    if task == "sentiment":
        print(pd.DataFrame(cm, index=common.SENTIMENT_NAMES, columns=common.SENTIMENT_NAMES))
        recall = common.per_class_recall(cm)
        for i, name in enumerate(common.SENTIMENT_NAMES):
            print(f"   {name:<9} recall {recall[i]:.3f}")
    else:
        print("   most common mistakes (true class, predicted class, count):")
        for true_c, pred_c, count in common.most_confused_pairs(cm, 5):
            print(f"      {true_c:2d} -> {pred_c:2d}   {count} times")

In [ ]:
for task in ["sentiment", "image"]:
    acc, settings, tree = best_trees[task]
    X_tr, y_tr, X_va, y_va = prepare(task)
    pred = tree.predict(X_va)
    np.save(f"decision_tree/outputs/{task}_val_pred.npy", pred)     # used by final/compare.ipynb
    print(f"\n{task}: best settings {settings}   validation accuracy {acc:.4f}")
    show_errors(task, y_va, pred)

## 6. Test predictions for Kaggle

Same settings, trained on **all** training rows.

In [ ]:
for task in ["sentiment", "image"]:
    acc, settings, _ = best_trees[task]
    X, y = DATA[task]
    X_test, test_ids = TEST[task]
    tree = DecisionTree(**settings).fit(X, y, N_CLASSES[task])
    pred = tree.predict(X_test)
    pd.DataFrame({"id": test_ids, "label": pred}).to_csv(f"decision_tree/outputs/{task}_test_pred.csv", index=False)
    print(f"{task}: saved decision_tree/outputs/{task}_test_pred.csv")

## 7. Notes for the presentation

* At which depth does the tree start to overfit?
* Why is one tree much weaker than k-NN or the neural network here? Each question looks at **one** feature only. The image task has 500 features and 50 classes, and the sentiment information is spread over many BERT features.
* Does `min_samples_leaf` or entropy change anything?